# View sample extractions from Qwen

Use the **extract** kernel. Run cells in order: give one paper UID, load its full Markdown, send it to Qwen with the sample-extraction prompt and compare the result with the samples already in the database. Prompts are editable below. Nothing is saved; the database and papers are only read.

## The 20 papers

These are the same 20 papers as in `synthesis_extraction.ipynb`. **The number of samples is from the previous extraction** (1–3 October 2026 run, read on 20 October 2026), not a corrected count.

| Paper UID | No. of samples | Title |
|---|---:|---|
| ID001100849 | 5 | Physics Augmented Machine Learning Discovery of Composition-Dependent Constitutive Laws for 3D Printed Digital Materials |
| ID001098910 | 7 | Polycaprolactone/graphite nanoplates composite nanopapers |
| ID001098286 | 6 | Interfacial Entropic Interactions Tunes Fragility and Dynamic Heterogeneity of Glassy Athermal Polymer Nanocomposite films |
| ID001098538 | 4 | Magnetic Shape Memory Polymers with Integrated Multifunctional Shape Manipulations |
| ID000573272 | 10 | Continuous glycolysis of waste PET via reactive extrusion and copolymerization with PTMG for enhanced toughness |
| ID000415115 | 6 | Effect of chemically modified lignin addition on the physicochemical properties of PCL nanofibers |
| ID000588793 | 11 | Structure-property relationships and microstructural origins of elasticity in Poly(ethylene brassylate)-block-poly(tetramethylene oxide) multiblock copolymers and their fibers |
| ID000581823 | 8 | Thermal properties, rheology and foams of polystyrene-block-poly(4-vinylpyridine) diblock copolymers |
| ID001002844 | 20 | Dispersant and Protective Roles of Amphiphilic Poly(ethylene phosphate) Block Copolymers in Polyester/Bone Mineral Composites. |
| ID001032337 | 6 | Biobased Thermoplastic Elastomers: Structure-Property Relationship of Poly(hexamethylene 2,5-furanodicarboxylate)-Block-Poly(tetrahydrofuran) Copolymers Prepared by Melt Polycondensation. |
| ID000752848 | 5 | Trans crystallization behavior and strong reinforcement effect of cellulose nanocrystals on reinforced poly(butylene succinate) nanocomposites. |
| ID000180835 | 2 | A comparative investigation of uranium and thorium adsorption behavior on amidoximated copolymeric hydrogel |
| ID000180254 | 6 | Flexible Conductive Nanocomposites for Electrochemical Devices Based on Chlorinated Natural Rubber/Nickel Oxide Nanoparticles |
| ID000750264 | 2 | A surface-grafted thermoresponsive hydrogel in which the surface structure dominates the bulk properties. |
| ID000732480 | 6 | Enhancing the degradability of poly(lactic acid) through the introduction of main-chain thionoester linkages by ring-opening copolymerization |
| ID000732356 | 16 | Ring-opening polymerization of six-membered cyclic hybrid dimers composed of an oxoester and thioester |
| ID000905310 | 24 | Green revolution in polymer composites: performance and degradability of natural fiber reinforced biopolymers |
| ID000137005 | 7 | A study of oriented conductive composites with segregated network structure obtained via solid-state processing of UHMWPE reactor powder and carbon nanofillers |
| ID000143254 | 4 | Synthesis of waterborne acrylic copolymer resin as a binding agent for the development of water-based inks in the printing application |
| ID000061162 | 4 | Antibacterial and thermomechanical properties of composites of polylactic acid modified with capsicum oleoresin-impregnated nanoporous silica |

In [ ]:
import sys
sys.dont_write_bytecode = True
import json
import sqlite3
import textwrap
from pathlib import Path

import pandas as pd
from IPython.display import display
from openai import AsyncOpenAI

# Find the repository from either notebooks/ or the repository root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/polymer_samples.db").is_file())

BASE_URL = "http://localhost:8000/v1"
MODEL = "qwen3.6-27b"
THINKING = True       # The 1–3 October run used reasoning on. Set False for thinking off.
MAX_TOKENS = 150_000  # Same cap as scripts/extract_samples_async.py.

## Prompts

Copied from the deployed sample extractor: `synth_extract/agents/extraction/prompts/sample/system_prompt.md`, `user_template.md` and `SampleExtractor.response_format()`. Edit these strings as needed; the files are not changed. Keep the `{markdown_full_text}` placeholder in `USER_PROMPT`.

In [ ]:
SYSTEM_PROMPT = """
Identify every distinct polymer material sample prepared in this paper and describe each one.

A sample is a polymer or polymer-based material made in this work. Samples are defined by how they were made: if the synthesis, formulation or processing differs (monomers, monomer ratio, composition, filler or additive loading, molecular weight, amounts, or reaction conditions such as temperature, time, catalyst, or solvent), it is a separate sample, even if the paper only reports it in a table or figure. 
Every row (entry) of a table that lists preparation conditions is a separate sample, also when the paper calls all of them by the same polymer name.
Do not merge samples that differ in synthesis. When in doubt, list it as its own sample.

A chemical modification of a sample produces a new sample. Test or measurement conditions do not create new samples.

DO NOT INCLUDE AS SAMPLES

- commercial materials used as received, only as constituents of another sample;
- individual monomers, initiators, catalysts, solvents, crosslinkers, additives, fillers, drugs, or other components when they are not themselves the sample;
- characterization instruments or measurement substrates;
- computational models;
- materials reported only from other papers;
- intermediates that are only used to prepare another sample (their synthesis belongs to that sample).

DESCRIPTION

The description should contain enough information to understand and distinguish the sample without reading the surrounding paper. Include, when reported:

1. what the material is: polymer chemistry, composition, architecture, or form;
2. how it was made: the key synthesis steps and conditions needed to identify it;
3. what distinguishes its synthesis from the other samples in the paper;
4. explicit values when they distinguish samples ("PLLA block Mn 20 kg/mol", not "longer PLLA block").

Every statement must come from the paper. Do not infer, generalize, or add background knowledge. If something is not mentioned in the paper, it does not go in the description.

FIELDS

- `name`: the polymer or material name as written in the paper (for example, "PEG-b-PLLA" or "PVDF membrane"). It is not a code or label. Different samples may share a name.
- `identifier`: the label the paper uses for this specific sample (for example, "PEL-10", "Sample 7", or "MN3"); use an empty string if none is given.
- `aliases`: every other name, code, or identifier the paper uses for this same sample.
- `description`: the self-contained description specified above.

If no qualifying samples are present, return an empty `samples` array. Return only the structured output.
"""

USER_PROMPT = """
<paper>
{markdown_full_text}
</paper>
"""

RESPONSE_FORMAT = {'type': 'json_schema',
 'json_schema': {'name': 'sample_list',
                 'strict': True,
                 'schema': {'type': 'object',
                            'properties': {'samples': {'type': 'array',
                                                       'items': {'type': 'object',
                                                                 'properties': {'name': {'type': 'string',
                                                                                         'minLength': 1},
                                                                                'identifier': {'type': 'string'},
                                                                                'aliases': {'type': 'array',
                                                                                            'items': {'type': 'string',
                                                                                                      'minLength': 1}},
                                                                                'description': {'type': 'string',
                                                                                                'minLength': 1}},
                                                                 'required': ['name',
                                                                              'identifier',
                                                                              'aliases',
                                                                              'description'],
                                                                 'additionalProperties': False}}},
                            'required': ['samples'],
                            'additionalProperties': False}}}

## Samples already in the database

Change `PAPER_UID` and run this cell. These are the records from the previous extraction, for comparison.

In [56]:
def get_samples(paper_uid):
    connection = sqlite3.connect(
        f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True
    )
    try:
        return pd.read_sql_query(
            "SELECT s.paper_uid, p.canonical_source AS source, s.sample_id, "
            "s.name, s.identifier, s.aliases, s.description "
            "FROM samples s JOIN papers p USING (paper_uid) "
            "WHERE s.paper_uid = ? ORDER BY s.sample_id",
            connection, params=(paper_uid,),
        )
    finally:
        connection.close()

PAPER_UID = "ID000061162"
db_samples = get_samples(PAPER_UID)
print(f"{PAPER_UID}: {len(db_samples)} samples in the database")
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(db_samples)

ID000061162: 4 samples in the database


,paper_uid,source,sample_id,name,identifier,aliases,description
0,ID000061162,wiley,830062,Neat PLA,PLA,"[""neat PLA""]","Poly(lactic acid) (PLA, grade Ingeo 2003D) dried overnight at 60 °C and compression molded at 1700 psi for 3 min at 180 °C without any filler."
1,ID000061162,wiley,830063,PLA/0.77 wt% SiCO composite,0.77 wt % SiCO,"[""0.77 wt % SiCO-modified PLA composite""]","PLA composite containing 0.77 wt% capsicum oleoresin-impregnated nanoporous silica (SiCO). Prepared by mixing dried PLA with 0.77 wt% SiCO at 170 °C for 10 min in an internal mixer at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C. The composite contains 0.5 wt% capsicum oleoresin (CO) relative to the total film weight."
2,ID000061162,wiley,830064,PLA/1.54 wt% SiCO composite,1.54 wt % SiCO,"[""1.54 wt % SiCO-modified PLA composite""]","PLA composite containing 1.54 wt% SiCO. Prepared by mixing dried PLA with 1.54 wt% SiCO at 170 °C for 10 min in an internal mixer at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C. The composite contains 1 wt% CO relative to the total film weight."
3,ID000061162,wiley,830065,PLA/3.08 wt% SiCO composite,3.08 wt % SiCO,"[""3.08 wt % SiCO-modified PLA composite""]","PLA composite containing 3.08 wt% SiCO. Prepared by mixing dried PLA with 3.08 wt% SiCO at 170 °C for 10 min in an internal mixer at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C. The composite contains 2 wt% CO relative to the total film weight."


## Choose a paper and load the full text

Set the paper UID here. This cell reads the paper's full local Markdown and builds `messages`. It does not call Qwen. You can inspect `paper_text` or `messages` before sending.

In [57]:
PAPER_UID = "ID000061162"

connection = sqlite3.connect(f"file:{ROOT / 'data/polymer_samples.db'}?mode=ro", uri=True)
try:
    found = connection.execute(
        "SELECT canonical_source, title FROM papers WHERE paper_uid = ?", (PAPER_UID,)
    ).fetchone()
finally:
    connection.close()
if found is None:
    raise ValueError(f"Paper {PAPER_UID} is not in the database.")
source, title = found

md_path = ROOT / "data/fulltext" / source / PAPER_UID / f"{PAPER_UID}.md"
paper_text = md_path.read_text(encoding="utf-8")
if not paper_text.strip():
    raise ValueError(f"The paper Markdown is empty: {md_path}")

user_message = USER_PROMPT.strip().replace("{markdown_full_text}", paper_text)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT.strip()},
    {"role": "user", "content": user_message},
]
print(f"{PAPER_UID} ({source}): {title}")
print(f"Loaded full Markdown: {len(paper_text):,} characters")

ID000061162 (wiley): Antibacterial and thermomechanical properties of composites of polylactic acid modified with capsicum oleoresin-impregnated nanoporous silica
Loaded full Markdown: 43,459 characters


## Send to Qwen and display the samples

Each execution makes one model request. The full response remains available as `response`; the samples are in `new_samples` (a DataFrame). Nothing is written to disk.

In [58]:
async with AsyncOpenAI(
    base_url=BASE_URL, api_key="none", timeout=1800, max_retries=0
) as client:
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.0,
        max_tokens=MAX_TOKENS,
        extra_body={"chat_template_kwargs": {"enable_thinking": THINKING}},
        response_format=RESPONSE_FORMAT,
    )

answer = response.choices[0].message.content or ""
new_samples = None
try:
    parsed = json.loads(answer)
    if isinstance(parsed, dict) and isinstance(parsed.get("samples"), list):
        new_samples = pd.DataFrame(
            parsed["samples"], columns=["name", "identifier", "aliases", "description"]
        )
except json.JSONDecodeError:
    pass

if new_samples is None:
    print(answer or "[No answer content returned]")
else:
    print(f"{PAPER_UID}: {len(new_samples)} samples extracted now")
    with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
        display(new_samples)
if response.choices[0].finish_reason != "stop":
    print("\nResponse may be incomplete. Finish reason:", response.choices[0].finish_reason)
usage = getattr(response, "usage", None)
if usage is not None:
    print(f"\nTokens: prompt {usage.prompt_tokens:,}, completion {usage.completion_tokens:,}")

ID000061162: 4 samples extracted now


,name,identifier,aliases,description
0,PLA,PLA,[neat PLA],Poly(lactic acid) (PLA) film (grade Ingeo 2003D). The PLA was dried overnight at 60 °C and then compression molded at 1700 psi for 3 min at 180 °C. This sample contains no filler or additive.
1,PLA/SiCO composite film,0.77 wt % SiCO,[0.77 wt % SiCO-modified PLA],"PLA composite film containing 0.77 wt % capsicum oleoresin-impregnated nanoporous silica (SiCO), corresponding to 0.5 wt % capsicum oleoresin (CO) in the final film. SiCO was prepared by mixing silica (35 wt %) with acetone at 50 °C for 10 min, adding extracted CO (65 wt %), stirring for 10 min at 50 °C, and drying at 60 °C for 2 h. The dried PLA was mixed with 0.77 wt % SiCO at 170 °C for 10 min in an internal mixer at 40 rpm, then compression molded at 1700 psi for 3 min at 180 °C."
2,PLA/SiCO composite film,1.54 wt % SiCO,[1.54 wt % SiCO-modified PLA],"PLA composite film containing 1.54 wt % SiCO, corresponding to 1 wt % CO in the final film. Prepared identically to the 0.77 wt % SiCO sample but with 1.54 wt % SiCO loading during the internal mixing step at 170 °C for 10 min at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C."
3,PLA/SiCO composite film,3.08 wt % SiCO,[3.08 wt % SiCO-modified PLA],"PLA composite film containing 3.08 wt % SiCO, corresponding to 2 wt % CO in the final film. Prepared identically to the other composites but with 3.08 wt % SiCO loading during the internal mixing step at 170 °C for 10 min at 40 rpm, followed by compression molding at 1700 psi for 3 min at 180 °C."



Tokens: prompt 14,252, completion 5,424


## Compare with the database

Lists the identifiers found now and in the previous extraction side by side. Run after both cells above for the same paper.

In [59]:
old = get_samples(PAPER_UID)
old_ids = [i or n for i, n in zip(old["identifier"], old["name"])]
new_ids = [] if new_samples is None else [
    i or n for i, n in zip(new_samples["identifier"], new_samples["name"])
]
rows = max(len(old_ids), len(new_ids))
comparison = pd.DataFrame({
    "previous extraction": old_ids + [""] * (rows - len(old_ids)),
    "now": new_ids + [""] * (rows - len(new_ids)),
})
print(f"previous: {len(old_ids)} samples, now: {len(new_ids)} samples")
with pd.option_context("display.max_rows", None):
    display(comparison)

previous: 4 samples, now: 4 samples


,previous extraction,now
0,PLA,PLA
1,0.77 wt % SiCO,0.77 wt % SiCO
2,1.54 wt % SiCO,1.54 wt % SiCO
3,3.08 wt % SiCO,3.08 wt % SiCO


1. for ID000732356, we see that earlier prompts squashed the fields and now we fixed them `Every row (entry) of a table that lists preparation conditions is a separate sample, also when the paper calls all of them by the same polymer name.`

## Reset the sample tables (fresh start)

**This cell writes to `data/polymer_samples.db`.** It deletes every row in `samples` and sets every paper in `sample_extraction_progress` back to `pending` (attempt count 0, errors and timestamps cleared), so the next extraction run extracts and stores all papers again. `papers` and `property_classification` are not changed.

- `CONFIRM_RESET` must be set to `True`, otherwise the cell stops without doing anything.
- `BACKUP_FIRST` writes a full copy of the database (`data/polymer_samples_backup_<date>_<time>.db`, about 1.3 GB) before deleting.
- `RESET_SAMPLE_ID_COUNTER` is off by default, so new samples get IDs after the old ones and no old `sample_id` in the synthesis runs or check sets can point to a different new sample.
- Stop all extraction jobs first; the cell refuses to run while any paper is `in_progress`.
- Deleted rows free space inside the file but do not shrink it; run `VACUUM` separately if needed.

In [ ]:
# Fresh start: delete every row in `samples` and set every paper in
# `sample_extraction_progress` back to 'pending', so the extraction jobs
# (scripts/extract_samples_async.py) extract and store all papers again.
# This WRITES to data/polymer_samples.db. The `papers` and
# `property_classification` tables are not touched.
import sqlite3
from contextlib import closing
from datetime import datetime

CONFIRM_RESET = False           # set to True to run the reset
BACKUP_FIRST = True             # copy the database before deleting anything
RESET_SAMPLE_ID_COUNTER = True # True: new samples start again at sample_id 1 (SID000001).
                                # Keep False if any file still refers to old sample_ids,
                                # otherwise an old ID could point to a different new sample.

samples_db_path = ROOT / "data" / "polymer_samples.db"
if not samples_db_path.is_file():
    raise FileNotFoundError(f"Database not found: {samples_db_path}")
if not CONFIRM_RESET:
    raise RuntimeError("Nothing done. Set CONFIRM_RESET = True to delete all samples and reset progress.")

def table_counts(conn):
    return {
        "samples": conn.execute("SELECT COUNT(*) FROM samples").fetchone()[0],
        "progress": dict(conn.execute(
            "SELECT status, COUNT(*) FROM sample_extraction_progress GROUP BY status"
        ).fetchall()),
        "next_sample_id": (conn.execute(
            "SELECT seq FROM sqlite_sequence WHERE name = 'samples'"
        ).fetchone() or (0,))[0] + 1,
    }

with closing(sqlite3.connect(samples_db_path, timeout=60)) as conn:
    before = table_counts(conn)
    if before["progress"].get("in_progress", 0):
        raise RuntimeError(
            f"{before['progress']['in_progress']} papers are 'in_progress'. "
            "Stop all extraction jobs before resetting."
        )

    if BACKUP_FIRST:
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_path = samples_db_path.with_name(f"polymer_samples_backup_{stamp}.db")
        if backup_path.exists():
            raise FileExistsError(f"Backup already exists: {backup_path}")
        with closing(sqlite3.connect(backup_path)) as backup:
            conn.backup(backup)
        print(f"Backup written: {backup_path}")

    conn.isolation_level = None  # manage the transaction explicitly
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("BEGIN IMMEDIATE")
    try:
        conn.execute("DELETE FROM samples")
        if RESET_SAMPLE_ID_COUNTER:
            conn.execute("DELETE FROM sqlite_sequence WHERE name = 'samples'")
        # Every paper gets a progress row, then every row goes back to 'pending'.
        conn.execute(
            "INSERT OR IGNORE INTO sample_extraction_progress (paper_uid) "
            "SELECT paper_uid FROM papers"
        )
        conn.execute(
            """
            UPDATE sample_extraction_progress
            SET status = 'pending',
                attempt_count = 0,
                last_error = NULL,
                started_at = NULL,
                completed_at = NULL,
                updated_at = CURRENT_TIMESTAMP
            """
        )
        conn.execute("COMMIT")
    except Exception:
        conn.execute("ROLLBACK")
        raise

    after = table_counts(conn)
    paper_count = conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0]

print(f"Database: {samples_db_path}")
print(f"samples rows: {before['samples']:,} -> {after['samples']:,}")
print(f"progress before: {before['progress']}")
print(f"progress after:  {after['progress']} / papers: {paper_count:,}")
print(f"next sample_id: {after['next_sample_id']:,}")

Backup written: /Users/kevinge/Work/Data Extraction/synth_extract/data/polymer_samples_backup_20261026_143352.db
Database: /Users/kevinge/Work/Data Extraction/synth_extract/data/polymer_samples.db
samples rows: 1,816,337 -> 0
progress before: {'failed': 4, 'success': 273187}
progress after:  {'pending': 273191} / papers: 273,191
next sample_id: 1
